# Bagging regressor
Single regressors against BaggingRegressor on the Boston housing data, then a grid search over the bagging
settings. The curve demo is `app.py` (`python app.py`, then open http://127.0.0.1:8050).

In [ ]:
import time

import pandas as pd
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

## 1. The data
`load_boston` was removed from scikit-learn in 1.2; the same table is in data/boston.csv (OpenML dataset 531).

In [ ]:
boston = pd.read_csv("data/boston.csv")
X = boston.drop(columns="MEDV")
y = boston["MEDV"]
print("inputs:", list(X.columns))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
print(X_train.shape, X_test.shape)

## 2. Three single regressors

In [ ]:
models = {"linear regression": LinearRegression(), "decision tree": DecisionTreeRegressor(random_state=1),
          "KNN": KNeighborsRegressor()}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: test R² {r2_score(y_test, model.predict(X_test)):.3f}")

## 3. A bagging regressor with every setting at its default
Default: 10 decision trees, each on a bootstrap sample of all the training rows, all columns.

In [ ]:
bag = BaggingRegressor(random_state=1)
bag.fit(X_train, y_train)
print(f"training R² {bag.score(X_train, y_train):.3f}")
print(f"test R²     {bag.score(X_test, y_test):.3f}")

## 4. Grid search: base model, number of models, row and column sampling
`estimator=None` means a decision tree. 3 x 3 x 2 x 2 x 2 x 2 = 144 combinations, 3-fold cross-validation.

In [ ]:
params = {
    "estimator": [None, LinearRegression(), KNeighborsRegressor()],
    "n_estimators": [20, 50, 100],
    "max_samples": [0.5, 1.0],
    "max_features": [0.5, 1.0],
    "bootstrap": [True, False],
    "bootstrap_features": [True, False],
}
start = time.time()
grid = GridSearchCV(BaggingRegressor(random_state=1), param_grid=params, cv=3, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
print(f"took {time.time() - start:.0f} s")
print(f"training R² {grid.best_estimator_.score(X_train, y_train):.3f}")
print(f"test R²     {grid.best_estimator_.score(X_test, y_test):.3f}")
print(f"best cross-validation R² {grid.best_score_:.3f}")
print("best parameters:", grid.best_params_)

## 5. Out-of-bag R²: the regressor also has oob_score

In [ ]:
bag_oob = BaggingRegressor(n_estimators=50, oob_score=True, random_state=1).fit(X_train, y_train)
print(f"out-of-bag R² {bag_oob.oob_score_:.3f}, test R² {bag_oob.score(X_test, y_test):.3f}")

## 6. The curve demo (same code as app.py)

In [ ]:
from app import figure, fit

(single, r1), (bagged, r2) = fit("decision tree", n_estimators=50, max_samples=25)
figure([(f"single tree: {r1:.2f}", single, "#E45756"), (f"bagging: {r2:.2f}", bagged, "#4C78A8")])